# Job Salary ML Model

The purpose of this program is to create a model that can predict the salary for job postings in the United States.

# Data Loading

### Data Paths

In [3]:
%spark
import org.apache.spark.sql.types._
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val ILOPath = "ILOData.parquet"
val OEPath = "cleanedOEData.parquet"
val LIPath = "LinkedinData.parquet"

val titleToSOCPath = "project/soc_2018_direct_match_title_file.csv"
val statePath = "project/states.csv"

In [4]:
%spark
val ILODF = spark.read
    .load(ILOPath)
    .withColumnRenamed("sex.label", "sex")
    .withColumnRenamed("classif1.label", "class")

z.show(ILODF)

In [5]:
%spark
val OEDF = spark.read.load(OEPath)

z.show(OEDF)

In [6]:
%spark
val LIDF = spark.read.load(LIPath)

z.show(LIDF)

### Employment Model

In [ ]:
%spark
val employmentOEDF = OEDF.filter($"areatype_name" === "National")
    .drop("areatype_name")
    .drop("area_name")
    .drop("state_name")
    
z.show(employmentOEDF)

### Wage Model

##### Set Up LI Wage

In [11]:
%spark
val wageLIDF = LIDF.select("title","location","normalized_salary","state")
    .withColumn("title", trim(col("title")))
    .filter($"normalized_salary".isNotNull)
    .filter($"state".isNotNull)

z.show(wageLIDF)

In [12]:
%spark
val titleToSOCDF = spark.read.format("csv")
    .option("inferSchema", "true")
    .option("header", "true")
    .load(titleToSOCPath)
    .select("2018 SOC Direct Match Title","2018 SOC Title")
    .withColumnRenamed("2018 SOC Direct Match Title","title")
    .withColumnRenamed("2018 SOC Title", "occupation_name")
    
z.show(titleToSOCDF)

In [13]:
%spark
val occupationCatOEDF = OEDF.filter($"occupation_description".isNotNull)
    .select("occupation_name","occupation_category_name","occupation_subcategory_name")
    .dropDuplicates("occupation_name")
    
z.show(occupationCatOEDF)

In [14]:
%spark
val titleToJobCatDF = titleToSOCDF.join(broadcast(occupationCatOEDF), Seq("occupation_name"))

z.show(titleToJobCatDF)

In [ ]:
%spark
val wageLIWithJobCatDF = wageLIDF.join(broadcast(titleToJobCatDF),Seq("title"))
    .withColumn("state", regexp_replace(col("state")," Metropolitan Area", ""))
    .withColumn("state", regexp_replace(col("state")," Area", ""))
    .filter($"state" =!= "United States")
    .filter($"normalized_salary" > 9999 && $"normalized_salary" < 1000001)
    .withColumn("log_salary", log($"normalized_salary"))
    .withColumnRenamed("location", "area_name")
    .withColumnRenamed("state", "state_name")
    .filter($"area_name".isNotNull)
    .filter($"state_name".isNotNull)
    .select("title","occupation_name","occupation_category_name","occupation_subcategory_name", "area_name","state_name", "normalized_salary", "log_salary")
    .withColumnRenamed("normalized_salary", "Annual Mean Wage")

z.show(wageLIWithJobCatDF)

##### Create Wage Model

In [ ]:
%spark
val wageOEDF = OEDF.filter($"areatype_name" === "Metropolitan or nonmetropolitan area")
    .drop("areatype_name")
    .drop("industry_name")
    .filter($"occupation_description".isNotNull)
    .drop("occupation_description")
    .filter($"datatype_name" === "Annual mean wage")
    .drop("datatype_name")
    .drop("sector_name")
    .withColumn("log_salary", log($"value"))
    .withColumnRenamed("value", "Annual Mean Wage")
    .withColumn("title", $"occupation_name")
    .select("title","occupation_name","occupation_category_name","occupation_subcategory_name", "area_name","state_name", "Annual Mean Wage", "log_salary")

z.show(wageOEDF)

In [18]:
%spark
trainwageOEDF.printSchema()

In [19]:
%spark
val combinedWageOELIDF = wageOEDF.unionByName(wageLIWithJobCatDF, allowMissingColumns=true)
val Array(trainWageOELIDF, testWageOELIDF) = combinedWageOELIDF.randomSplit(Array(.8, .2), seed=42)
println(f"There are ${trainWageOELIDF.cache().count()} rows in the training set, and ${testWageOELIDF.cache().count()} in the test set")

In [20]:
%spark
import org.apache.spark.ml.feature.{OneHotEncoder, StringIndexer}

val categoricalOELICols = trainWageOELIDF.dtypes.filter{
    case(field, dataType) => dataType == "StringType"
}.map(_._1)
val indexOutputOELICols = categoricalOELICols.map(_ + "_index")
val oheOutputOELICols = categoricalOELICols.map(_ + "_OHE")

val stringIndexerOELI = new StringIndexer()
  .setInputCols(categoricalOELICols)
  .setOutputCols(indexOutputOELICols)
  .setHandleInvalid("skip")

val oheEncoderOELI = new OneHotEncoder()
  .setInputCols(indexOutputOELICols)
  .setOutputCols(oheOutputOELICols)

In [21]:
%spark
import org.apache.spark.ml.feature.VectorAssembler

val vecAssemblerOELI = new VectorAssembler()
  .setInputCols(oheOutputOELICols)
  .setOutputCol("features")

In [22]:
%spark
import org.apache.spark.ml.regression.LinearRegression

val lrOELI = new LinearRegression()
  .setLabelCol("log_salary")
  .setFeaturesCol("features")

In [23]:
%spark
import org.apache.spark.ml.Pipeline

val pipelineOELI = new Pipeline()
  .setStages(Array(stringIndexerOELI, oheEncoderOELI, vecAssemblerOELI, lrOELI))

val pipelineModelOELI = pipelineOELI.fit(trainWageOELIDF)

In [24]:
%spark
val predOELIDF = pipelineModelOELI.transform(testWageOELIDF)
z.show(predOELIDF.select("features", "log_salary", "prediction"))

In [25]:
%spark
predOELIDF.printSchema()

In [26]:
%spark
val predFinalOELIDF = predOELIDF.withColumn("pred_salary", exp($"prediction"))
val predLIDF = predFinalOELIDF.select("title","occupation_name", "features","Annual Mean Wage","pred_salary").filter($"title" =!= $"occupation_name")

z.show(predLIDF)

In [27]:
%spark
z.show(predLIDF.select("title","occupation_name","Annual Mean Wage", "pred_salary").summary())

In [28]:
%spark
import org.apache.spark.ml.evaluation.RegressionEvaluator
val regressionEvaluatorLI = new RegressionEvaluator()
.setPredictionCol("pred_salary")
.setLabelCol("Annual Mean Wage")
.setMetricName("rmse")
val rmseLI = regressionEvaluatorLI.evaluate(predLIDF)

println(f"RMSE is $rmseLI%.1f")

In [29]:
%spark
val r2LI = regressionEvaluatorLI.setMetricName("r2").evaluate(predLIDF)
println(s"R2 is $r2LI")

In [ ]:
%spark
val statesDF = spark.read.format("csv")
    .option("inferSchema", "true")
    .option("header", "true")
    .load(statePath)
    .select("Abbreviation","State")
    
z.show(statesDF)